<a href="https://colab.research.google.com/github/al7arbi-111/strategy-reality-check/blob/main/reality_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd
from scipy import stats as st

def reality_check(returns, n_strategies_tried=1, n_params_tuned=0, cost_bps=0, rf=0.04):
    r = pd.Series(returns).dropna()
    n = len(r)
    years = n / 252
    ann_r = r.mean() * 252
    ann_v = r.std() * 252 ** 0.5
    sharpe = (ann_r - rf) / ann_v if ann_v > 0 else np.nan

    t = sharpe * np.sqrt(years)
    p = 2 * (1 - st.norm.cdf(abs(t)))

    skew = st.skew(r)
    kurt = st.kurtosis(r, fisher=False)

    return {
        "n_days": n, "years": round(years, 2),
        "sharpe": sharpe, "t_stat": t, "p_value": p,
        "skew": skew, "kurtosis": kurt,
        "ann_return": ann_r, "ann_vol": ann_v,
        "_r": r, "_tried": n_strategies_tried,
        "_params": n_params_tuned, "_cost": cost_bps,
    }

In [5]:
def deflated_sharpe(d):
    n, sr, tried = d["n_days"], d["sharpe"], d["_tried"]
    if tried <= 1:
        return sr, 0.0
    e = 0.5772156649
    expected_max = (1 - e) * st.norm.ppf(1 - 1/tried) + e * st.norm.ppf(1 - 1/(tried*np.e))
    sr_daily = sr / np.sqrt(252)
    haircut = expected_max * np.sqrt((1 + d["skew"]*sr_daily + (d["kurtosis"]-1)/4*sr_daily**2) / (n-1))
    return (sr_daily - haircut) * np.sqrt(252), haircut * np.sqrt(252)

In [8]:
def verdict(returns, name="strategy", n_strategies_tried=1, n_params_tuned=0, cost_bps=0):
    d = reality_check(returns, n_strategies_tried, n_params_tuned, cost_bps)
    dsr, haircut = deflated_sharpe(d)
    flags = []

    if d["years"] < 3:
        yrs = (1.96 / max(d["sharpe"], 0.01)) ** 2
        msg = (f"needs ~{int(np.ceil(yrs))}y at this Sharpe" if yrs < 50
               else "this Sharpe will never reach significance in a practical sample")
        flags.append(f"SHORT SAMPLE — {d['years']}y. {msg}.")

    if abs(d["t_stat"]) < 1.96:
        flags.append(f"NOT SIGNIFICANT — t={d['t_stat']:.2f}, p={d['p_value']:.3f}. Consistent with zero edge.")

    if d["_tried"] > 1:
        flags.append(f"SELECTION BIAS — {d['_tried']} strategies tried. Sharpe {d['sharpe']:.2f} deflates to {dsr:.2f} (haircut {haircut:.2f}).")

    if d["_params"] > 0:
        flags.append(f"PARAMETER TUNING — {d['_params']} parameter(s) fitted. Advisory only: not included in the deflation above, so the true haircut is larger than shown.")

    if d["skew"] < -0.5:
        flags.append(f"NEGATIVE SKEW — {d['skew']:.2f}. Small frequent gains, rare large losses.")

    if d["kurtosis"] > 5:
        flags.append(f"FAT TAILS — kurtosis {d['kurtosis']:.1f} vs 3 for normal. Sharpe understates tail risk.")

    if d["_cost"] == 0:
        flags.append("NO COSTS MODELLED — results are gross. Apply realistic costs before believing them.")

    print(f"=== {name} ===")
    print(f"  {d['n_days']} days ({d['years']}y) · return {d['ann_return']*100:.1f}% · vol {d['ann_vol']*100:.1f}% · Sharpe {d['sharpe']:.2f}")
    print()
    if flags:
        print(f"  {len(flags)} issue(s):")
        for f in flags:
            print(f"   • {f}")
    else:
        print("  No flags raised. This does not mean the strategy works.")
    print()

In [9]:
rng = np.random.default_rng(42)

pure_noise = pd.Series(rng.normal(0.0004, 0.01, 500))
verdict(pure_noise, "random noise, 2 years", n_strategies_tried=1)

best_of_50 = pd.Series(rng.normal(0.0009, 0.01, 500))
verdict(best_of_50, "best of 50 tested strategies", n_strategies_tried=50, n_params_tuned=2)

long_real = pd.Series(rng.normal(0.0005, 0.008, 3000))
verdict(long_real, "12-year track record", n_strategies_tried=1, cost_bps=5)

=== random noise, 2 years ===
  500 days (1.98y) · return 6.8% · vol 15.2% · Sharpe 0.18

  3 issue(s):
   • SHORT SAMPLE — 1.98y. this Sharpe will never reach significance in a practical sample.
   • NOT SIGNIFICANT — t=0.26, p=0.798. Consistent with zero edge.
   • NO COSTS MODELLED — results are gross. Apply realistic costs before believing them.

=== best of 50 tested strategies ===
  500 days (1.98y) · return 11.4% · vol 16.2% · Sharpe 0.46

  5 issue(s):
   • SHORT SAMPLE — 1.98y. needs ~19y at this Sharpe.
   • NOT SIGNIFICANT — t=0.65, p=0.518. Consistent with zero edge.
   • SELECTION BIAS — 50 strategies tried. Sharpe 0.46 deflates to -1.16 (haircut 1.61).
   • PARAMETER TUNING — 2 parameter(s) fitted. Advisory only: not included in the deflation above, so the true haircut is larger than shown.
   • NO COSTS MODELLED — results are gross. Apply realistic costs before believing them.

=== 12-year track record ===
  3000 days (11.9y) · return 9.3% · vol 12.7% · Sharpe 0.42

  1 